In [46]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import json
import pathlib
import dotenv
import pickle
import sys
import json5
import csv
import os
import pandas as pd
import numpy as np
from typing import Dict, List
from ast import literal_eval
from datetime import datetime
from pydantic import BaseModel, Field, ValidationError, parse_obj_as

current_dir = pathlib.Path().parent.resolve()
sys.path.append(str(current_dir.parent))

from loguru import logger
# from datahub_integrations.gen_ai.bedrock import BedrockModel
from datahub_integrations.gen_ai.term_suggestion_v2 import get_term_recommendations
from datahub_integrations.gen_ai.description_v2 import (
    extract_metadata_for_urn,
    transform_table_info_for_llm,
)
from datahub_integrations.gen_ai.term_suggestion_v2_context import (
    fetch_glossary_info,
    GlossaryUniverseConfig,
)
from docs_generation.graph_helper import create_datahub_graph
from term_suggestion_v2.helper import write_llm_output_to_csv

from term_suggestion_analysis_helper import (
    get_table_and_column_infos_dict, 
    get_failed_response_table_urns, 
    read_labeled_column_data,
    get_prediction_df,
    filter_predictions_df,
    get_classification_report_df
)
    
# TERM_SUGGESTION_GENERATION_MODEL: BedrockModel = parse_obj_as(
#     BedrockModel,
#     os.getenv(
#         "TERM_SUGGESTION_GENERATION_BEDROCK_MODEL", BedrockModel.CLAUDE_3_HAIKU.value
#     ),
# )

dotenv.load_dotenv(current_dir / ".env")

False

In [5]:
BASE_DIRECTORY = current_dir/ "term_suggestion_analysis"
COLUMN_LABELS_CSV_PATH = current_dir/ "column_labels_with_care_data.csv"
URNS_DICT_PATH = current_dir / "test_urns.json"

INSTANCES_TO_EXAMINE = ['longtailcompanions', 'acryl', 'chime', 'notion', 'twilio']


EXPERIMENT_NAME = "term_suggestion_output"
GLOSSARY_INSTANCE = "longtailcompanions"
GLOSSARY_NODES = ["urn:li:glossaryNode:PIITerms"]
GLOSSARY_TERMS = []
URNS_DICT: Dict[str, List[str]] = json5.loads(  # type: ignore
    (URNS_DICT_PATH).read_text()
)
    
FILTERS = [
    "no_filter", 
    "description", 
    "sample_values", 
    "description_and_sample_values", 
    "description_or_sample_values", 
    "name_and_datatype",
]
CONFIDENCE_THRESHOLDS = [7, 7.5, 8, 8.5, 9, 9.5]
    
URNS_DICT = {key:value for key, value in URNS_DICT.items() if key in INSTANCES_TO_EXAMINE}

In [8]:
table_infos_dict, column_infos_dict = get_table_and_column_infos_dict(urns_dict=URNS_DICT)

In [ ]:
# prompt = (current_dir / "../../src/datahub_integrations/gen_ai/term_suggestion_prompt.txt").read_text()
# prompt

# Fetch Glossary terms

In [52]:
graph_client = create_datahub_graph(GLOSSARY_INSTANCE)
glossary_config = GlossaryUniverseConfig(
    glossary_nodes=GLOSSARY_NODES,
    glossary_terms=GLOSSARY_TERMS
)
glossary_info = fetch_glossary_info(graph_client=graph_client, universe=glossary_config)

In [ ]:
print(len(glossary_info.glossary.keys()))
terms_to_remove = ['Day', 'Month', 'Year', 'Product ID', 'Country Code']
glossary_info.glossary = {key: value for key, value in glossary_info.glossary.items() if value["term_name"] not in terms_to_remove}
print(len(glossary_info.glossary.keys()))

In [53]:
# urns_dict_ = {key: value for key, value in urns_dict.items() if key in ['longtailcompanions', 'acryl']}

In [11]:
OUTPUT_DIR = BASE_DIRECTORY / f"{EXPERIMENT_NAME}_{datetime.now().strftime('%m-%d-%Y_%H-%M-%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pathlib.Path(OUTPUT_DIR / "prompt.txt").write_text(
    (
        current_dir / "../../src/datahub_integrations/gen_ai/term_suggestion_prompt.txt"
    ).read_text()
)


raw_llm_responses = []
parsed_llm_responses = []


for instance, urns in URNS_DICT.items():
    graph_client = create_datahub_graph(instance)
    for urn in urns:
        print(urn)
        try:
            table_terms, column_terms, raw_llm_response = get_term_recommendations(
                table_urn=urn, graph_client=graph_client, glossary_info=glossary_info
            )
            #             column_terms = label_fake_terms(column_terms)
            raw_llm_responses.append([urn, instance, raw_llm_response])
            parsed_llm_responses.append((urn, instance, table_terms, column_terms))
        except Exception as e:
            logger.exception(f"Exception Occurred {e}")
            raw_llm_responses.append([urn, instance, raw_llm_response])
            parsed_llm_responses.append((urn, instance, None, None))
            continue


write_llm_output_to_csv(llm_response=parsed_llm_responses, csv_path=OUTPUT_DIR / "parsed_response.csv")
with open(OUTPUT_DIR / "parsed_response.pkl", "wb") as fp:
    pickle.dump(parsed_llm_responses, fp)
    fp.close()

# Generate Analysis Reports

In [113]:
def func_categorize(row):
    if row["glossary_term"] is None and len(row["pred_max_score_term"]) == 0:
        return "match-no_assignment"
    elif (row["glossary_term"] in row["pred_max_score_term"]) or (
        row["Alternate_Glossary_Or_Comment"] in row["pred_max_score_term"]
    ):
        return "match-term_assigned"
    elif row["glossary_term"] is None and len(row["pred_max_score_term"]) != 0:
        return "mismatch-predicted_only"
    elif row["glossary_term"] is not None and len(row["pred_max_score_term"]) == 0:
        return "mismatch-actual_only"
    elif (row["glossary_term"] not in row["pred_max_score_term"]) and (
        row["Alternate_Glossary_Or_Comment"] not in row["pred_max_score_term"]
    ):
        return "mismatch"
    else:
        return "some_error"


def check_match(row):
    #     print(row)
    if (
        (row["glossary_term"] is None and len(row["pred_max_score_term"]) == 0)
        or row["glossary_term"] in row["pred_max_score_term"]
        or row["Alternate_Glossary_Or_Comment"] in row["pred_max_score_term"]
    ):
        return 1
    else:
        return 0

def get_merged_df(labeled_df, df_pred):
    merged_df = pd.merge(
        labeled_df,
        df_pred,
        on="unique_keys",
        how="left",
    )
    merged_df.loc[:, "pred_max_score_term"] = merged_df["pred_max_score_term"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    merged_df.loc[:, "predicted_labels"] = merged_df["predicted_labels"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    not_omitted_columns_df = merged_df[
        merged_df.unique_keys.isin(df_pred.unique_keys.tolist())
    ]
    omitted_columns_df = merged_df[
        ~merged_df.unique_keys.isin(df_pred.unique_keys.tolist())
    ]
    
    merged_df.loc[:, "match"] = merged_df.apply(lambda x: check_match(x), axis=1)
    merged_df.loc[:, "label_class"] = merged_df.apply(lambda x: func_categorize(x), axis=1)
    return merged_df, not_omitted_columns_df, omitted_columns_df

In [ ]:
# Confidence Thresolds
for CONFIDENCE_THRESHOLD in CONFIDENCE_THRESHOLDS:
    print("CONFIDENCE_THRESHOLD", CONFIDENCE_THRESHOLD)
    # Make Directory
    SUB_DIR = OUTPUT_DIR / f"threshold_{CONFIDENCE_THRESHOLD}"
    SUB_DIR.mkdir(parents=True, exist_ok=True)
    
    # Read labelled data
    labeled_df = read_labeled_column_data(COLUMN_LABELS_CSV_PATH, URNS_DICT)
    tables_with_parsing_error, skipped_tables = get_failed_response_table_urns(
        parsed_llm_responses
    )
    print("tables_with_parsing_error:", tables_with_parsing_error)
    print("skipped_tables:", skipped_tables)
    
    # Prepare prediction df:
    df_pred = get_prediction_df(parsed_llm_responses, CONFIDENCE_THRESHOLD)
    print("labeled_df", len(labeled_df))
    print("prediction_df", len(df_pred))
    
    # Merge Prediction df and labelled df
    merged_df, _, _ = get_merged_df(
        labeled_df[~labeled_df.table_urn.isin(tables_with_parsing_error)], 
        df_pred,
    )
    print("merged_df", len(merged_df))
    merged_df.to_csv(
        SUB_DIR / f"predicted_labels_threshold_{CONFIDENCE_THRESHOLD}.csv"
    )
    
    # Misclassification Analysis:
    miscalssification_analysis_cols = [
        "table_urn",
        "col_name",
        "col_description",
        "sample_values",
        "glossary_term",
        "Alternate_Glossary_Or_Comment",
        "predicted_labels",
    ]

    merged_df[merged_df.label_class == "mismatch-actual_only"][
        miscalssification_analysis_cols
    ].to_csv(SUB_DIR / f"FN_threshold_{CONFIDENCE_THRESHOLD}.csv")

    merged_df[merged_df.label_class == "mismatch-predicted_only"][
        miscalssification_analysis_cols
    ].to_csv(SUB_DIR / f"FP1_threshold_{CONFIDENCE_THRESHOLD}.csv")

    merged_df[merged_df.label_class == "mismatch"][miscalssification_analysis_cols].to_csv(
        SUB_DIR / f"FP2_threshold_{CONFIDENCE_THRESHOLD}.csv"
    )
    
    # Final Statistics
    final_stats = {}
    for filter_ in FILTERS:
        filtered_prediction_df = filter_predictions_df(merged_df, filter_)
        classification_report_df = get_classification_report_df(
            parsed_llm_responses=parsed_llm_responses,
            pred_df=filtered_prediction_df,
            column_infos_dict=column_infos_dict,
            labeled_df=labeled_df,
            filter_=filter_,
        )
        total_TP = classification_report_df.TP.sum()
        total_FP1 = classification_report_df.FP1.sum()
        total_FP2 = classification_report_df.FP2.sum()
        total_FN = classification_report_df.FN.sum()
        total_TN = classification_report_df.TN.sum()

        classification_report_df.to_csv(SUB_DIR / f"filter_{filter_}_classification_report_{CONFIDENCE_THRESHOLD}.csv")

        precision_term_for_all_tables = total_TP / (
            total_TP + total_FP1 + total_FP2
        ) if (total_TP + total_FP1 + total_FP2) != 0 else np.nan

        recall_term_for_all_tables = total_TP / (
            total_TP + total_FN + total_FP2
        ) if (total_TP + total_FN + total_FP2) != 0 else np.nan

        precision_none_for_all_tables = total_TN / (
            total_TN + total_FN
        ) if (total_TN + total_FN) != 0 else np.nan

        recall_none_for_all_tables = total_TN / (
            total_TN + total_FP1
        ) if (total_TN + total_FP1) != 0 else np.nan

        temp_stats = dict(
            classification_report_df[
                [
                    "fake_columns_count",
                    "fake_terms_count",
                    "actual_column_count",
                    "skipped_columns_count",
                    "TP",
                    "TN",
                    "FP1",
                    "FP2",
                    "FN",
                ]
            ].sum(axis=0)
        )
        temp_stats.update(
            {
                "precision_term_for_all_tables": precision_term_for_all_tables,
                "recall_term_for_all_tables": recall_term_for_all_tables,
                "precision_none_for_all_tables": precision_none_for_all_tables,
                "recall_none_for_all_tables": recall_none_for_all_tables,
                "threshold": CONFIDENCE_THRESHOLD,
            }
        )
        final_stats[filter_] = temp_stats  
    final_stats_df = pd.DataFrame(final_stats)
    final_stats_df.to_csv(SUB_DIR / f"final_stats_threshold_{CONFIDENCE_THRESHOLD}.csv") 

     **actual**            **predicted**           **cateory_name**
       None                    None                  match-no_assignment              TN
       None                    term                  mismatch-predicted_only          FP1
       term                    different term        mismatch                         FP2
       term                    term                  match-term_assigned              TP
       term                    None                  mismatch-actual_only             FN

    Recall of positive class:  TP/(TP + FN + FP2)
    Precision : TP/(TP + FP1 + FP2)

    Recal of negative class: TN/(TN + FP1)
    Precision of negative class: TN/(TN + FN)


In [ ]:
# no assignment- negative
# assignment - positive